# Load Data

In [1]:
import pandas as pd
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os
from pathlib import Path

# Extract credential access Spotify API

In [2]:
env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
REDIRECT_URI = os.getenv("REDIRECT_URI")

# Extract Spotify data

## Helper Functions

In [3]:
def extract_top_tracks_data(spotify_client,time_range)->pd.DataFrame:
    '''
    Convert JSON file extracted from spotify 
    to dataframe for analysis
    '''
    top_tracks_json = spotify_client.current_user_top_tracks(limit=50, time_range=time_range)
    track_list = []

    for item in top_tracks_json['items']:
        track_info = {
            'track_name':item['name'],
            'artist_name': item['artists'][0]['name'],
            'album_name': item['album']['name'],
            'release_date': item['album']['release_date'],
            'popularity': item['popularity'],
            'duration_ms': item['duration_ms'],
            'explicit':item['explicit'],
            'track_id': item['id']
        }
        track_list.append(track_info)
    
    data = pd.DataFrame(track_list)
    data['duration_min'] = data['duration_ms']/60000
    data['release_date'] = pd.to_datetime(data['release_date'])

    return data

def extract_top_artist_data(spotify_client,time_range)->pd.DataFrame:
    '''
    Convert JSON file extracted from Spotify
    to dataframe for analysis
    '''
    top_artist_json = spotify_client.current_user_top_artists(limit = 50,time_range = time_range)
    artist_list = []
    for item in top_artist_json['items']:
        artist_info = {
            'artist_name':item['name'],
            'popularity':item['popularity'],
            'genres': ', '.join(item['genres']),
            'artist_id': item['id'],
            # Gets the largest image if exists
            'image_url': item['images'][0]['url'] if item['images'] else None,
            'followers': item['followers']['total']
        }
        artist_list.append(artist_info)
    print(item['images'])
    data = pd.DataFrame(artist_list)
    return data

# Fetch Top Tracks

In [4]:
# Optional: remove existing cache for fresh login
if os.path.exists(".cache-my-music-app"):
    os.remove(".cache-my-music-app")

# Create the OAuth object
sp_oauth = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope=["user-top-read", "user-library-read", "playlist-read-private"],
    cache_path=".cache-my-music-app"
)

# Create the Spotify client using the SpotifyOAuth instance directly
sp = spotipy.Spotify(auth_manager=sp_oauth)

# Test API call
try:
    results = sp.current_user_top_tracks(limit=10)
    for idx, item in enumerate(results['items']):
        print(f"{idx+1}. {item['name']} - {item['artists'][0]['name']}")
except spotipy.exceptions.SpotifyException as e:
    print(f"Error: {e}")

print("Fetching your top tracks...")
top_tracks = sp.current_user_top_tracks(
    limit=50, time_range='medium_term'
    )

short_term_track_data = extract_top_tracks_data(sp,'short_term')
medium_term_track_data = extract_top_tracks_data(sp,'medium_term')
long_term_track_data = extract_top_tracks_data(sp,'long_term')

1. CASANOVA POSSE - ALI
2. HER - MINNIE
3. Imaginary Friend - ITZY
4. Air - YEJI
5. I TRUST YOU - エミリア(CV:高橋李依)
6. Shopper - IU
7. Secret - IU
8. Whiplash - aespa
9. FREAK - YUQI
10. NEMONEMO - YENA
Fetching your top tracks...


In [5]:
long_term_track_data

,track_name,artist_name,album_name,release_date,popularity,duration_ms,explicit,track_id,duration_min
0,Nobody - from Kaiju No. 8,OneRepublic,Nobody (from Kaiju No. 8),2024-04-12,70,153626,False,47N81NMkB488fuOwOC3Oip,2.560433
1,I GOT YOU,TWICE,I GOT YOU,2024-02-02,59,173240,False,35dhwUoJNlxrPyEIJkfDnx,2.887333
2,FREAK,YUQI,YUQ1,2024-04-23,57,171080,False,6ERs9uORCo1MfV0m9ixCuv,2.851333
3,DIVE,TWICE,DIVE,2024-07-10,57,181860,False,5vK3WrTOp6rEoASx1jAsp1,3.031000
4,Fate,(G)I-DLE,2,2024-01-29,62,161546,False,2vNPGH1x5ZwxTjlvzLCyc2,2.692433
5,Doughnut,TWICE,Celebrate,2022-07-27,53,263680,False,65rmgd5uMb4Rgqb5dSiU0p,4.394667
6,MORE & MORE,TWICE,MORE & MORE,2020-06-01,63,199653,False,3omvXShuRPM3zbDpWYqf5g,3.327550
7,Red Rover,YUQI,YUQ1,2024-04-23,48,123320,False,4TQBHR8LcbBUv0LvLmn54H,2.055333
8,Full Moon Full Life,高橋あず美,Persona 3 Reload Original Soundtrack,2024-04-24,65,293493,False,3Jl2LQmRwbXEF2lO1RTvxn,4.891550
9,YES or YES,TWICE,YES or YES,2018-11-05,68,237680,False,26OVhEqFDQH0Ij77QtmGP9,3.961333


In [6]:
short_term_track_data

,track_name,artist_name,album_name,release_date,popularity,duration_ms,explicit,track_id,duration_min
0,Air,YEJI,Air,2025-03-10,55,194800,False,2HhIndg75YiKjuUgGiMjSA,3.246667
1,Draw the Moon (feat. MIYAVI),MINNIE,"Webtoon <Myst, Might, Mayhem> OST Part. 2 Draw...",2025-03-21,45,210146,False,4B3JCEcAeTofpsfsEianeS,3.502433
2,Radio (Dum-Dum),YUQI,Radio (Dum-Dum),2025-03-17,67,152373,False,0mXXjVVAhaasNXga2HMgJK,2.539550
3,I,TAEYEON,I - The 1st Mini Album,2015-10-07,61,206038,False,5ZkITfPpcNPnyYGTibkO6m,3.433967
4,Invasion,YEJI,Air,2025-03-10,40,167800,False,3ePablAj7jk2c1j5CKEtAv,2.796667
5,HAPPY,DAY6,Fourever,2024-03-18,64,189934,False,1k68vKHNQXU5CHqcM7Yp7N,3.165567
6,"Can’t Slow Me, No",YEJI,Air,2025-03-10,38,178573,False,7EbaA7z7wP3G7xfYZVlVJS,2.976217
7,Shopper,IU,The Winning,2024-02-20,55,215720,False,1c6kkrWnpy68eYDfBdxNtF,3.595333
8,SMILEY-Japanese Ver.- (feat.ちゃんみな),YENA,SMILEY-Japanese Ver.- (feat.ちゃんみな),2023-08-07,39,176882,False,4RhkH4fGvvEzxvRlUcYH9L,2.948033
9,NEMONEMO,YENA,NEMONEMO,2024-09-30,67,178026,False,4UwsXGVppRRJpKBHy0mtyK,2.967100


# Fetch Top Artist

In [7]:
extract_top_artist_data(sp,'short_term')

[{'height': 640, 'url': 'https://i.scdn.co/image/ab6761610000e5ebf9736f54357de6014f551fca', 'width': 640}, {'height': 320, 'url': 'https://i.scdn.co/image/ab67616100005174f9736f54357de6014f551fca', 'width': 320}, {'height': 160, 'url': 'https://i.scdn.co/image/ab6761610000f178f9736f54357de6014f551fca', 'width': 160}]


,artist_name,popularity,genres,artist_id,image_url,followers
0,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,https://i.scdn.co/image/ab6761610000e5ebf7a109...,8355136
1,YENA,56,k-pop,49muoiIu4uea4PO8vueUNN,https://i.scdn.co/image/ab6761610000e5eb1135e1...,806960
2,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,https://i.scdn.co/image/ab6761610000e5ebbd0642...,9133293
3,TAEYEON,67,"k-pop, k-ballad",3qNVuliS40BLgXGxhdBdqu,https://i.scdn.co/image/ab6761610000e5ebb5d9eb...,3064356
4,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,https://i.scdn.co/image/ab6761610000e5eb1a9a17...,760546
5,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,https://i.scdn.co/image/ab6761610000e5ebca6c14...,21661915
6,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,https://i.scdn.co/image/ab6761610000e5eb344806...,8349525
7,YEJI,61,k-pop,3skli1w2n0nOZ4qkDbvV2m,https://i.scdn.co/image/ab6761610000e5eb485cfb...,113471
8,YUQI,60,k-pop,22aCD8IrQZjcPgZw728QT6,https://i.scdn.co/image/ab6761610000e5eb569026...,832868
9,NMIXX,71,k-pop,28ot3wh4oNmoFOdVajibBl,https://i.scdn.co/image/ab6761610000e5ebb75519...,3605836


In [8]:
extract_top_artist_data(sp,'long_term')

[{'height': 640, 'url': 'https://i.scdn.co/image/ab6761610000e5ebb256ae9a4b82bfff97776ae2', 'width': 640}, {'height': 320, 'url': 'https://i.scdn.co/image/ab67616100005174b256ae9a4b82bfff97776ae2', 'width': 320}, {'height': 160, 'url': 'https://i.scdn.co/image/ab6761610000f178b256ae9a4b82bfff97776ae2', 'width': 160}]


,artist_name,popularity,genres,artist_id,image_url,followers
0,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,https://i.scdn.co/image/ab6761610000e5ebca6c14...,21661915
1,(G)I-DLE,73,k-pop,2AfmfGFbe0A0WsTYm0SDTx,https://i.scdn.co/image/ab6761610000e5eb7fd163...,10487023
2,Ed Sheeran,89,soft pop,6eUKZXaKkcviH0Ku9w2n3V,https://i.scdn.co/image/ab6761610000e5eb784daf...,119909905
3,Taylor Swift,98,,06HL4z0CvFAxyc27GXpf02,https://i.scdn.co/image/ab6761610000e5ebe672b5...,135368184
4,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,https://i.scdn.co/image/ab6761610000e5ebbd0642...,9133293
5,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,https://i.scdn.co/image/ab6761610000e5eb344806...,8349525
6,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,https://i.scdn.co/image/ab6761610000e5ebf7a109...,8355136
7,Against The Current,64,pop punk,6yhD1KjhLxIETFF7vIRf8B,https://i.scdn.co/image/ab6761610000e5eb469aba...,528861
8,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,https://i.scdn.co/image/ab6761610000e5eb1a9a17...,760546
9,IVE,75,k-pop,6RHTUrRF63xao58xh9FXYJ,https://i.scdn.co/image/ab6761610000e5eb538dad...,5448007


In [9]:
extract_top_artist_data(sp,'medium_term')

[{'height': 640, 'url': 'https://i.scdn.co/image/ab6761610000e5eb469aba515a0815396f1c432c', 'width': 640}, {'height': 320, 'url': 'https://i.scdn.co/image/ab67616100005174469aba515a0815396f1c432c', 'width': 320}, {'height': 160, 'url': 'https://i.scdn.co/image/ab6761610000f178469aba515a0815396f1c432c', 'width': 160}]


,artist_name,popularity,genres,artist_id,image_url,followers
0,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,https://i.scdn.co/image/ab6761610000e5ebf7a109...,8355136
1,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,https://i.scdn.co/image/ab6761610000e5ebbd0642...,9133293
2,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,https://i.scdn.co/image/ab6761610000e5eb1a9a17...,760546
3,YUQI,60,k-pop,22aCD8IrQZjcPgZw728QT6,https://i.scdn.co/image/ab6761610000e5eb569026...,832868
4,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,https://i.scdn.co/image/ab6761610000e5ebca6c14...,21661915
5,(G)I-DLE,73,k-pop,2AfmfGFbe0A0WsTYm0SDTx,https://i.scdn.co/image/ab6761610000e5eb7fd163...,10487023
6,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,https://i.scdn.co/image/ab6761610000e5eb344806...,8349525
7,IVE,75,k-pop,6RHTUrRF63xao58xh9FXYJ,https://i.scdn.co/image/ab6761610000e5eb538dad...,5448007
8,YENA,56,k-pop,49muoiIu4uea4PO8vueUNN,https://i.scdn.co/image/ab6761610000e5eb1135e1...,806960
9,Ed Sheeran,89,soft pop,6eUKZXaKkcviH0Ku9w2n3V,https://i.scdn.co/image/ab6761610000e5eb399444...,119909905
